# DPR processor example with Prefect+Dask

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-520

See the associated Python module: [dpr_processor_example.py](./dpr_processor_example.py)

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
init_dask_cluster_eopf(scale=4)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/3cc7f3fdbcb04f6a81834e860721e61f/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [3]:
%%bash
prefect block ls
echo -e "\nNOTE: you can see the block details and credentials by running e.g.: prefect block inspect s3-bucket/s3"

                                     Blocks                                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ ID                                  ┃ Type     ┃ Name     ┃ Slug             ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ 2966ec95-af0b-4196-bc0e-40399a9c71… │ S3 Buck… │ user-s3  │ s3-bucket/user-… │
│ 1b8601a2-6fff-4a2c-99b1-b202b51076… │ Secret   │ dask-au… │ secret/dask-auth │
└─────────────────────────────────────┴──────────┴──────────┴──────────────────┘
                 List Block Types using `prefect block type ls`                 

NOTE: you can see the block details and credentials by running e.g.: prefect block inspect s3-bucket/s3


In [4]:
# Other imports
import getpass
import os
from importlib import reload
from pathlib import Path
from resources import prefect_utils

# Test the DPR processing with n dummy products
output_count = 3
s3_basename = "new_zarr_product_"
s3_filenames = [f"{s3_basename}{i}" for i in range(output_count)]

# Data to test the example flow. 
# Depending on the functions we call, we need to pass a different dir (yikes)
s3_subdir = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/zarr"
s3_prefix_subdir = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_subdir}"
s3_full_path = f"s3://{PREFECT_BLOCK_S3.bucket_name}/{s3_prefix_subdir}"
my_data = {"s3_folder": s3_full_path, "s3_filenames": s3_filenames}

# Data as a command-line string in the json format
my_data_str = json.dumps(my_data)
my_data_str = my_data_str.replace('"', r'\"')

print(f"Output zarr products will be written to: {s3_full_path}")

Output zarr products will be written to: s3://prefect-share/sub/dir/users/jgaucher/zarr


In [5]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

# Set environment variables for the client and dask workers
def set_dask_env():
    os.environ["HELLO_FROM"] = "dask"
dask_client.run(set_dask_env)
os.environ["HELLO_FROM"] = "client"

In [6]:
# NOTE: we need to create the S3 folder with a dummy file before running DPR
empty_file = "/tmp/.empty"
!echo "empty" > $empty_file
await PREFECT_BLOCK_S3.upload_from_path(empty_file, f"{s3_subdir}/{empty_file}")

I0000 00:00:1738839546.084012    2290 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


10:59:06.384 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/.empty' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/zarr/tmp/.empty'.

'sub/dir/users/jgaucher/zarr/tmp/.empty'

# 1. Call Prefect flow from Python code
This is useful to debug or see the generated HTML representation, but in production we'll deploy the flow (see next section).

In [7]:
print(f"Remove existing zarr products from: {s3_full_path!r}")
PREFECT_BLOCK_S3._get_bucket_resource().objects.filter(Prefix=f"{s3_prefix_subdir}/{s3_basename}").delete()

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/zarr'


[{'ResponseMetadata': {'RequestId': '1821995ACFDF91E8',
   'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'HTTPStatusCode': 200,
   'HTTPHeaders': {'accept-ranges': 'bytes',
    'content-length': '650',
    'content-type': 'application/xml',
    'server': 'MinIO',
    'strict-transport-security': 'max-age=31536000; includeSubDomains',
    'vary': 'Origin, Accept-Encoding',
    'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
    'x-amz-request-id': '1821995ACFDF91E8',
    'x-content-type-options': 'nosniff',
    'x-ratelimit-limit': '4944',
    'x-ratelimit-remaining': '4944',
    'x-xss-protection': '1; mode=block',
    'date': 'Thu, 06 Feb 2025 10:59:06 GMT'},
   'RetryAttempts': 0},
  'Deleted': [{'Key': 'sub/dir/users/jgaucher/zarr/new_zarr_product_0.zarr/.zattrs'},
   {'Key': 'sub/dir/users/jgaucher/zarr/new_zarr_product_0.zarr/.zgroup'},
   {'Key': 'sub/dir/users/jgaucher/zarr/new_zarr_product_1.zarr/.zattrs'},
  

In [8]:
# Import the module, or reload it if you changed its source code
import dpr_processor_example
reload(dpr_processor_example)

# Run the flow
results = dpr_processor_example.dpr_flow(**my_data)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.2  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


10:59:06.734 | WARNING | root - Hello from 'client' '172.18.0.21' (main code)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.2  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


10:59:06.829 | WARNING | root - Hello from 'client' '172.18.0.21' (main code)

10:59:06.890 | INFO    | prefect.engine - Created flow run 'space-stallion' for flow 'dpr-flow'

...

10:59:16.941 | ERROR   | Flow run 'space-stallion' - timed out after 10 s.

10:59:16.943 | INFO    | Flow run 'space-stallion' - Try #2

10:59:19.502 | INFO    | Flow run 'space-stallion' - Try #1

10:59:19.515 | INFO    | Flow run 'space-stallion' - Try #1

10:59:19.560 | INFO    | Flow run 'space-stallion' - Finished in state Completed()

In [9]:
# Display HTML representation
import IPython
for result in results:
    display(IPython.display.HTML(result))
del results

In [10]:
# Download zarr products into local
local_dir = "/tmp/zarr/"
!rm -rf "$local_dir" && mkdir -p "$local_dir"
await PREFECT_BLOCK_S3.get_directory(f"{s3_prefix_subdir}", local_dir)
!ls -al "$local_dir"

# Open them with the zarr python package
!pip install zarr
import zarr

total 24
drwxr-xr-x 6 jovyan users 4096 Feb  6 10:59 .
drwxrwxrwt 1 root   root  4096 Feb  6 10:59 ..
-rw-r--r-- 1 jovyan users    0 Feb  6 10:59 .empty
drwxr-xr-x 3 jovyan users 4096 Feb  6 10:59 new_zarr_product_0.zarr
drwxr-xr-x 3 jovyan users 4096 Feb  6 10:59 new_zarr_product_1.zarr
drwxr-xr-x 3 jovyan users 4096 Feb  6 10:59 new_zarr_product_2.zarr
drwxr-xr-x 2 jovyan users 4096 Feb  6 10:59 tmp


In [11]:
# see: https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data
for filename in s3_filenames:
    store = zarr.open(f"{local_dir}/{filename}.zarr")
    display(store.tree())
    
    # Read some data
    zarr_array = store["measurements"]["image"]["sensor1"]
    display(zarr_array)
    
    # Read the data into memory as a NumPy array
    numpy_array = zarr_array[:]
    display(numpy_array)

/
└── measurements
    ├── image
    │   ├── oa1_radiance (1024, 1024) int64
    │   ├── sensor1 (1024, 1024) int64
    │   └── sensor2 (1024, 1024) int64
    └── orphans
        └── orphans_oa01_radiance (1024, 1024) int64

<Array file:///tmp/zarr/new_zarr_product_0.zarr/measurements/image/sensor1 shape=(1024, 1024) dtype=int64>

array([[21, 65, 77, ...,  4, 79, 24],
       [44, 65, 67, ..., 27, 92,  9],
       [98, 60, 20, ..., 49,  9, 22],
       ...,
       [42, 30, 38, ..., 19,  2, 22],
       [13, 44, 88, ..., 56,  5, 23],
       [59, 79, 40, ..., 64, 93, 61]], shape=(1024, 1024))

/
└── measurements
    ├── image
    │   ├── oa1_radiance (1024, 1024) int64
    │   ├── sensor1 (1024, 1024) int64
    │   └── sensor2 (1024, 1024) int64
    └── orphans
        └── orphans_oa01_radiance (1024, 1024) int64

<Array file:///tmp/zarr/new_zarr_product_1.zarr/measurements/image/sensor1 shape=(1024, 1024) dtype=int64>

array([[56, 53, 33, ..., 96,  1, 28],
       [65, 16,  9, ..., 22, 81, 43],
       [90, 35, 22, ..., 69, 61,  4],
       ...,
       [ 3, 29, 82, ..., 94, 24,  7],
       [ 4, 27,  8, ..., 54, 39, 71],
       [ 4, 69, 79, ..., 81, 13,  6]], shape=(1024, 1024))

/
└── measurements
    ├── image
    │   ├── oa1_radiance (1024, 1024) int64
    │   ├── sensor1 (1024, 1024) int64
    │   └── sensor2 (1024, 1024) int64
    └── orphans
        └── orphans_oa01_radiance (1024, 1024) int64

<Array file:///tmp/zarr/new_zarr_product_2.zarr/measurements/image/sensor1 shape=(1024, 1024) dtype=int64>

array([[94, 34, 57, ..., 36, 52, 37],
       [26, 12, 42, ..., 29, 95, 26],
       [68, 16, 25, ..., 92, 34, 43],
       ...,
       [36, 49, 53, ..., 68, 28, 99],
       [79, 10, 29, ..., 43, 61,  0],
       [ 3, 77, 24, ..., 63, 36,  1]], shape=(1024, 1024))

# 2. Deploy Prefect flow

See the full yaml file: [deploy-prefect-dask.yaml](./deploy-prefect-dask.yaml)

Deploy our source code via the S3 bucket.

In [12]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://prefect-share/sub/dir/users/jgaucher/code'


In [13]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./dpr_processor_example.yaml"

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.2  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
10:59:23.640 | WARNING | root - Hello from 'client' '172.18.0.21' (main code)


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'dpr-flow/sprint20-dpr-example' successfully created with id      │
│ '486dcb52-4878-4517-959c-700083345bc5'.                                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/486dcb52-4878-4517-959c-700083345bc5


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 'dpr-flow/sprint20-dpr-example'



In [14]:
deploy_name = "dpr-flow/sprint20-dpr-example"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'dpr-flow/sprint20-dpr-example'


In [15]:
print(f"Remove existing zarr products from: {s3_full_path!r}")
PREFECT_BLOCK_S3._get_bucket_resource().objects.filter(Prefix=f"{s3_prefix_subdir}/{s3_basename}").delete()

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/zarr'


[{'ResponseMetadata': {'RequestId': '1821995EEE502739',
   'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'HTTPStatusCode': 200,
   'HTTPHeaders': {'accept-ranges': 'bytes',
    'content-length': '7163',
    'content-type': 'application/xml',
    'server': 'MinIO',
    'strict-transport-security': 'max-age=31536000; includeSubDomains',
    'vary': 'Origin, Accept-Encoding',
    'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
    'x-amz-request-id': '1821995EEE502739',
    'x-content-type-options': 'nosniff',
    'x-ratelimit-limit': '4944',
    'x-ratelimit-remaining': '4944',
    'x-xss-protection': '1; mode=block',
    'date': 'Thu, 06 Feb 2025 10:59:24 GMT'},
   'RetryAttempts': 0},
  'Deleted': [{'Key': 'sub/dir/users/jgaucher/zarr/new_zarr_product_0.zarr/.zattrs'},
   {'Key': 'sub/dir/users/jgaucher/zarr/new_zarr_product_0.zarr/.zgroup'},
   {'Key': 'sub/dir/users/jgaucher/zarr/new_zarr_product_0.zarr/.zmetadata'}

In [16]:
%%bash -s "$deploy_name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 'dpr-flow/sprint20-dpr-example'...
Created flow run 'ambitious-mongrel'.
└── UUID: 7f371584-527a-4114-883c-804ede874960
└── Parameters: {'s3_folder': 's3://prefect-share/sub/dir/users/jgaucher/zarr', 's3_filenames': ['new_zarr_product_0', 'new_zarr_product_1', 'new_zarr_product_2']}
└── Job Variables: {}
└── Scheduled start time: 2025-02-06 10:59:25 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/7f371584-527a-4114-883c-804ede874960
Watching flow run 'ambitious-mongrel'...


10:59:25.468 | INFO    | prefect - Flow run is in state 'Scheduled'


10:59:30.284 | WARNING | Flow run 'ambitious-mongrel' - Hello from 'dask' '172.18.0.3' (new_zarr_product_0)

10:59:30.284 | WARNING | Flow run 'ambitious-mongrel' - Hello from 'dask' '172.18.0.3' (new_zarr_product_0)

10:59:30.284 | WARNING | Flow run 'ambitious-mongrel' - Hello from 'dask' '172.18.0.3' (new_zarr_product_0)

...

10:59:30.549 | ERROR   | Task run 'single_dpr_task' - Finished in state Failed("Task run encountered an exception ContainsGroupError: path 'measurements' contains a group")

10:59:30.549 | ERROR   | Task run 'single_dpr_task' - Finished in state Failed("Task run encountered an exception ContainsGroupError: path 'measurements' contains a group")

10:59:35.512 | INFO    | prefect - Flow run is in state 'Failed'


Flow run finished in state 'Failed'.


CalledProcessError: Command 'b'# Trigger a run for this flow from the command line\nprefect deployment run "$1" --params "$2" --watch\n'' returned non-zero exit status 1.

In [ ]:
# Download zarr products into local (same as above)
local_dir = "/tmp/zarr/"
!rm -rf "$local_dir" && mkdir -p "$local_dir"
await PREFECT_BLOCK_S3.get_directory(f"{s3_prefix_subdir}", local_dir)
!ls -al "$local_dir"

In [ ]:
# see: https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data
for filename in s3_filenames:
    store = zarr.open(f"{local_dir}/{filename}.zarr")
    display(store.tree())
    
    # Read some data
    zarr_array = store["measurements"]["image"]["sensor1"]
    display(zarr_array)
    
    # Read the data into memory as a NumPy array
    numpy_array = zarr_array[:]
    display(numpy_array)

## 3. Shutdown the dask clusters

In [ ]:
# You can scale the clusters to 0 workers
dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.